# 05 — Training Loop

Loss functions, optimizers, LR schedulers, and the standard train/validate loop with metric tracking, checkpointing, and early stopping.

## Loss Functions

In [ ]:
import torch
import torch.nn as nn

nn.CrossEntropyLoss()      # multi-class classification (expects raw logits + class indices)
nn.BCELoss()                # binary classification (expects probabilities in [0,1])
nn.BCEWithLogitsLoss()      # binary classification (expects raw logits, numerically stable)
nn.MSELoss()                 # regression
nn.L1Loss()                  # regression, robust to outliers
nn.SmoothL1Loss()            # Huber loss, used in detection tasks
nn.NLLLoss()                 # paired with LogSoftmax output
nn.KLDivLoss()                # distribution matching


## Optimizers

In [ ]:
from torch.optim import SGD, Adam, AdamW, RMSprop

model = nn.Linear(10, 2)

SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999))
AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)   # decoupled weight decay
RMSprop(model.parameters(), lr=1e-3)


## LR Schedulers

In [ ]:
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, ReduceLROnPlateau, OneCycleLR

optimizer = Adam(model.parameters(), lr=1e-3)

StepLR(optimizer, step_size=10, gamma=0.1)                 # decay every N epochs
CosineAnnealingLR(optimizer, T_max=50)                     # smooth cosine decay
ReduceLROnPlateau(optimizer, mode="min", patience=3)       # decay when val loss stalls
OneCycleLR(optimizer, max_lr=1e-2, total_steps=1000)       # warmup + decay, popular for fast training


## Full Training Loop (with validation, best-checkpoint, early stopping)

In [ ]:
import copy

def train_model(model, train_loader, val_loader, epochs=20, lr=1e-3, patience=5, device="cpu"):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        # --- train ---
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        # --- validate ---
        model.eval()
        val_loss, correct = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * xb.size(0)
                correct += (preds.argmax(1) == yb).sum().item()
        val_loss /= len(val_loader.dataset)
        val_acc = correct / len(val_loader.dataset)

        scheduler.step(val_loss)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"epoch {epoch+1}/{epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

        # --- checkpoint + early stopping ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    return model, history


## Gradient Accumulation (simulate a larger batch size)

In [ ]:
accumulation_steps = 4
optimizer.zero_grad()
for i, (xb, yb) in enumerate(train_loader):
    preds = model(xb)
    loss = criterion(preds, yb) / accumulation_steps
    loss.backward()
    if (i + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()


## Metric Tracking (confusion matrix, precision/recall via torchmetrics-style manual calc)

In [ ]:
def accuracy(preds, targets):
    return (preds.argmax(1) == targets).float().mean().item()

def confusion_matrix(preds, targets, num_classes):
    preds = preds.argmax(1)
    cm = torch.zeros(num_classes, num_classes, dtype=torch.int64)
    for t, p in zip(targets.view(-1), preds.view(-1)):
        cm[t.long(), p.long()] += 1
    return cm


## Common Gotchas
- Not calling `optimizer.zero_grad()` each step accumulates stale gradients.
- Computing loss/accuracy in train mode vs eval mode gives different numbers if using Dropout/BatchNorm — always `model.eval()` + `torch.no_grad()` for validation.
- `ReduceLROnPlateau.step(val_loss)` needs the metric passed in, unlike other schedulers which take no args.
- Averaging loss across batches should weight by batch size if the last batch is smaller (`drop_last=True` avoids this).